# 🎶 YuE2-Studio (Colab & Kaggle Edition)
Dashboard generasi musik **YuE2** + **LoRA Training Studio** untuk **Google Colab** dan **Kaggle** (kompatibel dengan GPU **T4 16 GB** / **P100**), terinspirasi dari **YuE2-Studio**.

- **Inference Engine:** `timoncool/yue2.cpp` (`yue-server` CUDA + Q8_0 GGUF)
- **LoRA Trainer:** `timoncool/HOT-Step-CPP` (`ace-train` CUDA)
- **Dashboard:** Gradio Bilingual (ID/EN) + opsional tunnel WebUI bawaan `yue2.cpp`

In [ ]:
#@title 1. Deteksi Platform (Colab / Kaggle), Cek GPU & Mount Google Drive
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER = "/content/drive/MyDrive/YuE2-Studio" #@param {type:"string"}

import os, pathlib
!nvidia-smi

IS_KAGGLE = pathlib.Path("/kaggle/working").exists()
IS_COLAB = pathlib.Path("/content").exists() and not IS_KAGGLE
BASE_ROOT = pathlib.Path("/kaggle/working") if IS_KAGGLE else pathlib.Path("/content")
WORKSPACE = BASE_ROOT / "yue2-workspace"

for sub in ["models", "adapters", "library", "datasets", "trained_loras", "bin"]:
    (WORKSPACE / sub).mkdir(parents=True, exist_ok=True)

if IS_KAGGLE:
    USE_GOOGLE_DRIVE = False
    print(f"🟢 Terdeteksi: KAGGLE | Workspace: {WORKSPACE}")
elif IS_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_root = pathlib.Path(DRIVE_FOLDER)
    for sub in ["bin", "models", "adapters", "library", "datasets", "trained_loras"]:
        (drive_root / sub).mkdir(parents=True, exist_ok=True)
    print(f"🟢 Terdeteksi: GOOGLE COLAB | Drive Cache: {drive_root}")
else:
    print(f"ℹ️ Workspace lokal: {WORKSPACE}")

os.environ["YUE_WORKSPACE"] = str(WORKSPACE)
os.environ["YUE_DRIVE_ROOT"] = DRIVE_FOLDER

In [ ]:
#@title 2. Install Dependensi & Build yue2.cpp (yue-server CUDA)
import pathlib, shutil

YUE_BIN = WORKSPACE / "bin/yue-server"
DRIVE_YUE_BIN = pathlib.Path(DRIVE_FOLDER) / "bin/yue-server"

if USE_GOOGLE_DRIVE and DRIVE_YUE_BIN.exists() and not YUE_BIN.exists():
    print("📦 Memulihkan yue-server dari cache Google Drive...")
    shutil.copy2(DRIVE_YUE_BIN, YUE_BIN)
    YUE_BIN.chmod(0o755)

if not YUE_BIN.exists():
    !apt-get update -qq && apt-get install -y -qq cmake ninja-build libcurl4-openssl-dev ffmpeg
    SRC_YUE = BASE_ROOT / "yue2.cpp"
    if not SRC_YUE.exists():
        !git clone --recurse-submodules https://github.com/timoncool/yue2.cpp.git "{SRC_YUE}"
    !cmake -S "{SRC_YUE}" -B "{SRC_YUE}/build" -G Ninja -DGGML_CUDA=ON -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc -DCMAKE_CUDA_ARCHITECTURES="60;70;75;80;86;89"
    !cmake --build "{SRC_YUE}/build" --config Release -j $(nproc)
    shutil.copy2(SRC_YUE / "build/yue-server", YUE_BIN)
    if USE_GOOGLE_DRIVE:
        shutil.copy2(YUE_BIN, DRIVE_YUE_BIN)
        print("💾 yue-server disimpan ke Google Drive cache.")
else:
    print(f"✅ yue-server siap di {YUE_BIN}")

In [ ]:
#@title 3. Build HOT-Step-CPP Trainer (ace-train CUDA untuk LoRA Training)
BUILD_LORA_TRAINER = True #@param {type:"boolean"}
import os, pathlib, shutil

TRAIN_BIN = WORKSPACE / "bin/ace-train"
DRIVE_TRAIN_BIN = pathlib.Path(DRIVE_FOLDER) / "bin/ace-train"

if BUILD_LORA_TRAINER:
    if USE_GOOGLE_DRIVE and DRIVE_TRAIN_BIN.exists() and not TRAIN_BIN.exists():
        print("📦 Memulihkan ace-train dari cache Google Drive...")
        shutil.copy2(DRIVE_TRAIN_BIN, TRAIN_BIN)
        TRAIN_BIN.chmod(0o755)

    if not TRAIN_BIN.exists():
        SRC_TRAIN = BASE_ROOT / "HOT-Step-CPP"
        if not SRC_TRAIN.exists():
            !git clone --recurse-submodules https://github.com/timoncool/HOT-Step-CPP.git "{SRC_TRAIN}"
        !cmake -S "{SRC_TRAIN}/engine" -B "{SRC_TRAIN}/engine/build" -G Ninja -DGGML_CUDA=ON -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc -DCMAKE_CUDA_ARCHITECTURES="60;70;75;80;86;89"
        !cmake --build "{SRC_TRAIN}/engine/build" --target ace-train --config Release -j $(nproc)
        built_bin = SRC_TRAIN / "engine/build/ace-train"
        if built_bin.exists():
            shutil.copy2(built_bin, TRAIN_BIN)
            if USE_GOOGLE_DRIVE:
                shutil.copy2(TRAIN_BIN, DRIVE_TRAIN_BIN)
                print("💾 ace-train disimpan ke Google Drive cache.")
    os.environ["ACE_TRAIN_BIN"] = str(TRAIN_BIN)
    print(f"✅ Trainer siap di {TRAIN_BIN}")

In [ ]:
#@title 4. Download Model YuE2 (Q8_0 Backbone + VAE F32)
QUANT = "Q8_0" #@param ["Q8_0", "Q6_K", "Q5_K_M", "BF16"]
!pip install -q gradio>=4.44.0 requests>=2.31.0 huggingface_hub>=0.25.0 pydantic>=2.7.0

import pathlib, shutil
from huggingface_hub import hf_hub_download

MODELS_DIR = WORKSPACE / "models"
DRIVE_MODELS = pathlib.Path(DRIVE_FOLDER) / "models"

for fname in [f"YuE2-3B-{QUANT}.gguf", "YuE2-Vae-F32.gguf"]:
    local_f = MODELS_DIR / fname
    drive_f = DRIVE_MODELS / fname
    if USE_GOOGLE_DRIVE and drive_f.exists() and not local_f.exists():
        print(f"📦 Menyalin {fname} dari Google Drive...")
        shutil.copy2(drive_f, local_f)
    elif not local_f.exists():
        print(f"⬇️ Mengunduh {fname} dari Serveurperso/YuE2-GGUF...")
        hf_hub_download(repo_id="Serveurperso/YuE2-GGUF", filename=fname, local_dir=str(MODELS_DIR))
        if USE_GOOGLE_DRIVE:
            shutil.copy2(local_f, drive_f)
    print(f"✅ {fname} siap.")

os.environ["YUE_MODEL_FILE"] = str(MODELS_DIR / f"YuE2-3B-{QUANT}.gguf")
os.environ["YUE_VAE_FILE"] = str(MODELS_DIR / "YuE2-Vae-F32.gguf")

In [ ]:
#@title 5. Jalankan yue-server & YuE2-Studio Dashboard (Gradio)
GITHUB_REPO = "https://github.com/USERNAME/yuestudio.git" #@param {type:"string"}
import os, pathlib, subprocess, time, requests

REPO_DIR = BASE_ROOT / "yuestudio"
if not (REPO_DIR / "dashboard/app.py").exists():
    drive_repo = pathlib.Path(DRIVE_FOLDER) / "yuestudio"
    kaggle_in = pathlib.Path("/kaggle/input/yuestudio")
    if GITHUB_REPO and "USERNAME" not in GITHUB_REPO:
        print(f"⬇️ Cloning repo dari GitHub: {GITHUB_REPO}")
        !git clone --depth 1 "{GITHUB_REPO}" "{REPO_DIR}"
    elif USE_GOOGLE_DRIVE and (drive_repo / "dashboard/app.py").exists():
        !cp -r "{drive_repo}" "{REPO_DIR}"
    elif (kaggle_in / "dashboard/app.py").exists():
        !cp -r "{kaggle_in}" "{REPO_DIR}"
    else:
        raise FileNotFoundError(f"Isi GITHUB_REPO di atas dengan URL repo GitHub kamu, atau upload folder 'dashboard/' ke {REPO_DIR}")

server_cmd = [
    str(WORKSPACE / "bin/yue-server"),
    "--host", "0.0.0.0",
    "--port", "8087",
    "--model", os.environ["YUE_MODEL_FILE"],
    "--vae", os.environ["YUE_VAE_FILE"],
    "--adapters", str(WORKSPACE / "adapters"),
]
log_f = open(str(BASE_ROOT / "yue-server.log"), "w")
server_proc = subprocess.Popen(server_cmd, stdout=log_f, stderr=subprocess.STDOUT)

print("⏳ Menunggu yue-server aktif di port 8087...")
for _ in range(45):
    try:
        if requests.get("http://127.0.0.1:8087/health", timeout=2).status_code == 200:
            print("🟢 yue-server ONLINE!")
            break
    except Exception:
        pass
    time.sleep(2)

os.environ["GRADIO_SHARE"] = "1"
%cd "{REPO_DIR}"
!python -m dashboard.app

In [ ]:
#@title 6. (Opsional) Tunnel WebUI Bawaan yue2.cpp (Port 8087 via Cloudflared)
#@markdown Jalankan sel ini hanya jika ingin mengakses WebUI asli bawaan `yue2.cpp`.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared tunnel --url http://127.0.0.1:8087